# 00 — Inspect the source data

**What we are doing:** Before running regressions, inspect whether we have 12 months, four destination CBGs, and reasonable data types. This is quality control, not a model.

**Privacy:** Keep the Dewey ZIP in `data/private/`, not in the public GitHub repo.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PRIVATE = ROOT / 'data' / 'private'
with zipfile.ZipFile(PRIVATE / 'ECON499_Dewey_2024_4Malls.zip') as archive:
    flows = pd.read_csv(archive.open('Origin_Destination_Flows.csv'), dtype={'Origin_CBG':str, 'Destination_CBG':str})
    monthly = pd.read_csv(archive.open('Mall_Monthly_Summary.csv'), dtype={'Destination_CBG':str})
print('Monthly summary rows:', len(monthly))
print('Origin flow rows:', len(flows))
print('Malls:', sorted(monthly.Mall.unique()))
print('Months:', sorted(monthly.Month.unique()))
print('Unique origin CBGs:', flows.Origin_CBG.nunique())
assert len(monthly) == 48 and len(flows) == 95803
assert monthly.Month.nunique() == 12 and monthly.Mall.nunique() == 4

## Annual summary
**What we are doing:** Add the 12 monthly counts for each mall-containing CBG. **These are not unique annual shoppers.**

In [ ]:
annual = monthly.groupby('Mall', as_index=False).agg(
    annual_device_counts=('Device_Counts','sum'),
    annual_stop_counts=('Stop_Counts','sum'))
print(annual.to_string(index=False))
ax = monthly.pivot(index='Month',columns='Mall',values='Device_Counts').plot(marker='o',figsize=(10,5))
ax.set(title='Monthly device activity in destination CBGs',ylabel='Reported devices (not mall admissions)',xlabel='2024 month')

## Validate other sources
**What we are doing:** Check that the ACS file has 25 origin areas and the distance workbook has exactly 25 × 4 = 100 routes.

In [ ]:
acs = pd.read_csv(PRIVATE/'ECON499_ACS_2024_25_ZCTA_Checked.csv',dtype={'ZCTA':str})
distance = pd.read_excel(PRIVATE/'ECON499_100_Driving_Distances_RETESTED.xlsx',sheet_name='100 Routes',dtype={'Origin ZCTA':str})
print('ACS ZCTAs:',acs.ZCTA.nunique(),'distance rows:',len(distance))
print('Distance columns:',list(distance.columns))
assert acs.ZCTA.nunique()==25 and len(distance)==100